# Dataset Loading from Google Drive

In [ ]:
# Install gdown to download files from Google Drive
!pip install gdown -q

In [ ]:
import gdown
import pandas as pd
import os
import json

# Extract the file ID from the Google Drive link
file_id = '1mqcTmnScJCmS62qoSXjAZ-YqkxuXNl__'
output_file_name = 'raw_dataset'

# Download the file
try:
    gdown.download(id=file_id, output=output_file_name, quiet=False, fuzzy=True)
    print(f"File downloaded successfully to: {output_file_name}")

    # Check file type and load into DataFrame
    if os.path.exists(output_file_name):
        print(f"Inspecting the first 5 lines of '{output_file_name}':")
        with open(output_file_name, 'r') as f:
            for i, line in enumerate(f):
                if i >= 5:
                    break
                print(f"Line {i+1}: {line.strip()}")

        try:
            with open(output_file_name, 'r') as f:
                data = json.load(f)
            print("File loaded as JSON object successfully.")
            print("Top-level keys in the JSON data:", data.keys())


            try:
                df = pd.json_normalize(data)
                print("Dataset loaded into DataFrame using json_normalize.")
            except Exception as e_normalize:
                print(f"Could not load file into DataFrame using json_normalize: {e_normalize}")
                print("Unsupported JSON structure for direct DataFrame conversion. Please inspect the keys and manually extract the data.")
                df = None

        except Exception as e_json_load:
            print(f"Could not load file as JSON object: {e_json_load}")
            print("Unsupported file format or invalid JSON. Please inspect the contents.")
            df = None

        if df is not None:
            print("Dataset loaded successfully. Displaying the first 5 rows:")
            display(df.head())
    else:
        print("Download failed: file not found locally after gdown.")

except Exception as e:
    print(f"An error occurred during download: {e}")
    print("Please ensure the Google Drive file is shared with 'Anyone with the link'.")

Downloading...
From: https://drive.google.com/uc?id=1mqcTmnScJCmS62qoSXjAZ-YqkxuXNl__
To: /content/raw_dataset
100%|██████████| 179k/179k [00:00<00:00, 66.6MB/s]

File downloaded successfully to: raw_dataset
Inspecting the first 5 lines of 'raw_dataset':
Line 1: {
Line 2: "metadata": {
Line 3: "vocabulary_size": 1500,
Line 4: "test_case_count": 1000,
Line 5: "total_corpus_words": 735040
File loaded as JSON object successfully.
Top-level keys in the JSON data: dict_keys(['metadata', 'word_counts', 'test_cases'])
Dataset loaded into DataFrame using json_normalize.
Dataset loaded successfully. Displaying the first 5 rows:


,test_cases,metadata.vocabulary_size,metadata.test_case_count,metadata.total_corpus_words,word_counts.the,word_counts.of,word_counts.and,word_counts.to,word_counts.a,word_counts.in,...,word_counts.buy,word_counts.shelter,word_counts.drawn,word_counts.dust,word_counts.communism,word_counts.exchange,word_counts.sections,word_counts.walls,word_counts.foot,word_counts.aircraft
0,[{'input': 'itthatthecitytakestepstothisproble...,1500,1000,735040,69971,36412,28853,26158,23195,21337,...,70,70,70,70,70,70,70,70,70,70


In [ ]:
df['test_cases']

,test_cases
0,[{'input': 'itthatthecitytakestepstothisproble...


In [ ]:
import json

with open("/content/raw_dataset", "r", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))

if isinstance(data, dict):
    print(data.keys())
else:
    print(len(data))
    print(data[:2])

<class 'dict'>
dict_keys(['metadata', 'word_counts', 'test_cases'])


In [ ]:
import pprint

pprint.pp(data[1] if isinstance(data, list) else data)

In [ ]:
print(data.keys())

dict_keys(['metadata', 'word_counts', 'test_cases'])


In [ ]:
list(data['metadata'].items())[:5]

[('vocabulary_size', 1500),
 ('test_case_count', 1000),
 ('total_corpus_words', 735040)]

In [ ]:
frq_dict=data['word_counts']
list(frq_dict.items())[:5]

[('the', 69971), ('of', 36412), ('and', 28853), ('to', 26158), ('a', 23195)]

In [ ]:
vocab=list(data['word_counts'].keys())
vocab[:5]


['the', 'of', 'and', 'to', 'a']

In [ ]:
test_cases=data['test_cases']

In [ ]:
dummy_ip=list(data['test_cases'])[0]
dummy_ip

{'input': 'itthatthecitytakestepstothisproblem',
 'ground_truth': 'it that the city take steps to this problem',
 'word_count': 9}

In [ ]:
maxLengthWord = max(len(word) for word in vocab)
maxLengthWord

14

In [ ]:
def cal_accuracy(pred_tokens, op_tokens):
    correct = 0
    maxLengthWord = max(len(word) for word in vocab)
    for pred, actual in zip(pred_tokens, op_tokens):
        if pred == actual:
            correct += 1

    return correct / len(op_tokens)

In [ ]:
def cal_edit_dist(pred_tokens, op_tokens):
    wrong = 0

    for pred, actual in zip(pred_tokens, op_tokens):
        if pred != actual:
            wrong  += 1

    return wrong

### Implementing Greedy Based Approach

In [ ]:
def my_greedy(text,vocab,frq_dict):
  ip_text=text['input']

  op_text_ideal=text['ground_truth']
  op_text_tokens=text['ground_truth'].split()
  op_text_number = text['word_count']
  greedy_tokens=list()
  j = 0 # Initialize j outside the loop, and update it correctly
  while j < len(ip_text):
    max_l_word=min([maxLengthWord,len(ip_text[j:])])
    flag=0
    for l in range(max_l_word,0,-1):
      if ip_text[j:j+l] in vocab:
        greedy_tokens.append(ip_text[j:j+l])
        j=j+l
        flag = 1
        break
    if(flag==0):

      greedy_tokens.append(ip_text[j])
      j=j+1

  greedy_text_number = len(greedy_tokens)
  accuracy_score=cal_accuracy(greedy_tokens,op_text_tokens)
  edit_dist_score = cal_edit_dist(greedy_tokens, op_text_tokens)
  pred_text=" ".join(greedy_tokens)
  return (pred_text,accuracy_score,edit_dist_score)

#### Test greedy

In [ ]:
pred_text,acc,edit_d=my_greedy(dummy_ip,vocab,frq_dict)
pred_text,acc,edit_d

('it that the city takes t e p s to this problem', 0.4444444444444444, 5)

In [ ]:
final_acc=0
final_edit_d=0
count=0
for ip in test_cases:
  pred_text,acc,edit_d=my_greedy(ip,vocab,frq_dict)
  final_acc=final_acc+acc
  final_edit_d = final_edit_d+edit_d
  count=count+1
print(f"Accuracy of Greedy Approch is {final_acc/count} and avg. edit dist is {final_edit_d/count}")

Accuracy of Greedy Approch is 0.8343584054834053 and avg. edit dist is 1.472


### Dynamic programming approach

In [ ]:
import math
def prob_of_word(word,frq_dict):
  if word not in frq_dict.keys():
    return 0
  word_count=frq_dict[word]
  total_count=sum(frq_dict.values())
  return round(word_count/total_count,7)


prob_dict = {x: prob_of_word(x, frq_dict) for x in frq_dict.keys()}
list(prob_dict.items())[:5]

[('the', 0.0951935),
 ('of', 0.0495374),
 ('and', 0.0392536),
 ('to', 0.0355872),
 ('a', 0.0315561)]

In [ ]:
def log_prob(word):
  num = prob_of_word(word,frq_dict)
  return round(math.log(num,10),7) # 10 base log

log_prob_dict = {x: log_prob(x) for x in frq_dict.keys()}
list(log_prob_dict.items())[:5]

[('the', -1.0213927),
 ('of', -1.3050668),
 ('and', -1.4061205),
 ('to', -1.4487062),
 ('a', -1.5009167)]

$$DP[i] = \max_{0 \le j < i} \left( DP[j] + \log P(\text{text}[j:i]) \right)$$

In [ ]:
def my_dp(text, vocab, frq_dict, prob_dict, log_prob_dict):

    ip_text = text['input']
    op_text_ideal = text['ground_truth']
    op_text_tokens = text['ground_truth'].split()
    op_text_number = text['word_count']

    # DP arrays
    dp = [-float('inf')] * (len(ip_text) + 1)
    parent = [-1] * (len(ip_text) + 1)

    dp[0] = 0

    # DP
    for i in range(1, len(ip_text) + 1):

        for j in range(0, i):

            word = ip_text[j:i]

            if word in vocab:

                score = dp[j] + log_prob_dict[word]

                if score > dp[i]:
                    dp[i] = score
                    parent[i] = j


    dp_tokens = []
    # Generate word tokens from last
    i = len(ip_text)

    while i > 0:

        j = parent[i]

        word = ip_text[j:i]

        dp_tokens.append(word)

        i = j

    dp_tokens.reverse() # to get last word token as 1st one

    # Calculate metrics
    accuracy_score = cal_accuracy(dp_tokens, op_text_tokens)
    edit_dist_score = cal_edit_dist(dp_tokens, op_text_tokens)

    pred_text = " ".join(dp_tokens)

    return (pred_text, accuracy_score, edit_dist_score)

#### Testing DP

In [ ]:
pred_text,acc,edit_d=my_dp(dummy_ip,vocab,frq_dict,prob_dict,log_prob_dict)
pred_text,acc,edit_d

('it that the city take steps to this problem', 1.0, 0)

In [ ]:
final_acc=0
final_edit_d=0
count=0
for ip in test_cases:
  pred_text,acc,edit_d=my_dp(ip,vocab,frq_dict,prob_dict,log_prob_dict)
  final_acc=final_acc+acc
  final_edit_d = final_edit_d+edit_d
  count=count+1
print(f"Accuracy of DP Approch is {final_acc/count} and avg. edit dist is {final_edit_d/count}")

Accuracy of DP Approch is 0.9903365079365078 and avg. edit dist is 0.062


* For DP:
  * A very common word can have a high probability, so DP may prefer several common words over the actual less-common word.